In [53]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer # used to fill the empty values 
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder , LabelEncoder

In [ ]:
df = pd.read_csv("covid_toy.csv")
df.sample(5)

,age,gender,fever,cough,city,has_covid
75,5,Male,102.0,Mild,Kolkata,Yes
76,80,Male,100.0,Mild,Bangalore,Yes
68,54,Female,104.0,Strong,Kolkata,No
92,82,Female,102.0,Strong,Kolkata,No
3,31,Female,98.0,Mild,Kolkata,No


In [6]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

### Manually doing the encoding for all the columns seperatey


In [ ]:
si = SimpleImputer()  # this class is used to fill the missing values in the columns
X_train_fever = si.fit_transform(X_train[['fever']])

# same for test data
X_test_fever = si.fit_transform(X_test[['fever']])

In [17]:
X_train_fever.shape

(80, 1)

In [ ]:
# ordinal encoding -> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape, X_test_cough.shape

((80, 1), (20, 1))

In [ ]:
# oneHot encoding -> gender,city
ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# test data
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

X_train_gender_city.shape, X_test_gender_city.shape

((80, 4), (20, 4))

In [ ]:
# extracting Age
X_train_age = X_train.drop(columns=['gender', 'city', 'cough', 'fever']).values

# test data
X_test_age = X_test.drop(columns=['gender', 'city', 'cough', 'fever']).values

In [ ]:
# now concat all the data encoded together using numpy

X_train_transform = np.concatenate(
    (X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)

# test data
X_test_transform = np.concatenate(
    (X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

X_train_transform.shape, X_test_transform.shape

((80, 7), (20, 7))

### Now we will encode all he columns together using ColumneTransformer

In [41]:
from sklearn.compose import ColumnTransformer

In [42]:
# drop will delete the column and passthrough will not change anything in the column
transformer = ColumnTransformer(transformers=[
    # how many encoding to do we will pass it into an tuple , we are using 3 therefor 3 tuple
    # each tuple include (tansformer_name,Transformer_object(),list of column name where that encoding to apply)
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first',sparse_output=False),['gender','city'])
    ],
    remainder='passthrough')

In [47]:
transformer.fit_transform(X_train)
transformer.fit_transform(X_train).shape

(80, 7)

In [46]:
transformer.fit_transform(X_test)
transformer.fit_transform(X_test).shape

(20, 7)

In [51]:
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

adult_df = pd.read_csv("adult.data", header=None, names=columns)


In [62]:
trans = ColumnTransformer(transformers=[
    ('tnf1',LabelEncoder(),['income']),
    
    ('tnf2',OrdinalEncoder(
    categories=[[   
    ' Preschool',
    ' 1st-4th',
    ' 5th-6th',
    ' 7th-8th',
    ' 9th',
    ' 10th',
    ' 11th',
    ' 12th',
    ' HS-grad',
    ' Some-college',
    ' Assoc-voc',
    ' Assoc-acdm',
    ' Bachelors',
    ' Masters',
    ' Prof-school',
    ' Doctorate']
    ]),['education']),
    
    ('tnf3',OneHotEncoder(drop='first',sparse_output=False),["workclass","marital-status","occupation","relationship","race","sex","native-country"])
    ],
    remainder='passthrough')